In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import pygeohash as pgh
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor
from pathlib import Path
from itertools import combinations

In [2]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

In [3]:
train_df = train_df.dropna()

In [4]:
train_df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy
5,5,qp02zw,48,0:0,0.016262,Residential,2,Not Allowed,Yes,8.446025,Rainy
6,6,qp02zy,48,0:0,0.042247,Residential,3,Allowed,Yes,15.772408,Foggy


In [5]:
geo_map = {
    gh: pgh.decode(gh)
    for gh in train_df['geohash'].unique()
}

train_df['lat'] = train_df['geohash'].map(
    lambda x: geo_map[x][0]
)

train_df['lon'] = train_df['geohash'].map(
    lambda x: geo_map[x][1]
)

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
df = train_df.copy()
# ======================================
# LAT/LON FEATURES
# ======================================

# Fit scaler on latitude and longitude
scaler = StandardScaler()

scaled_coords = scaler.fit_transform(
    df[['lat', 'lon']]
)

# Create scaled features
df['lat_scaled'] = scaled_coords[:, 0]
df['lon_scaled'] = scaled_coords[:, 1]

# Distance from dataset center
df['distance_from_city_center'] = np.sqrt(
    df['lat_scaled']**2 +
    df['lon_scaled']**2
)

# Drop original coordinates
df.drop(columns=['lat', 'lon'], inplace=True)

# ======================================
# TIMESTAMP FEATURES
# ======================================

# Split HH:MM
df['hour'] = df['timestamp'].str.split(':').str[0].astype(int)
df['minute'] = df['timestamp'].str.split(':').str[1].astype(int)

# Minutes since midnight
df['minutes_since_midnight'] = (
    df['hour'] * 60 +
    df['minute']
)

# Cyclic encoding
df['time_sin'] = np.sin(
    2 * np.pi * df['minutes_since_midnight'] / 1440
)

df['time_cos'] = np.cos(
    2 * np.pi * df['minutes_since_midnight'] / 1440
)

# Drop original timestamp
df.drop(columns=['timestamp'], inplace=True)

# ======================================
# BINARY FEATURES
# ======================================

df['Landmarks'] = df['Landmarks'].map({
    'No': 0,
    'Yes': 1
})

df['LargeVehicles'] = df['LargeVehicles'].map({
    'Not Allowed': 0,
    'Allowed': 1
})

# ======================================
# ONE HOT ENCODING
# ======================================

df = pd.get_dummies(
    df,
    columns=['RoadType', 'Weather'],
    dummy_na=False,
    drop_first=True,
    dtype=int
)

# ======================================
# CHECK RESULT
# ======================================

print(df.shape)
print(df.head())
print(df.dtypes)

(73459, 21)
   Index geohash  day    demand  NumberofLanes  LargeVehicles  Landmarks  \
1      1  qp02zt   48  0.118507              3              1          1   
2      2  qp08bj   48  0.027132              1              0          0   
4      4  qp02zq   48  0.010819              1              0          0   
5      5  qp02zw   48  0.016262              2              0          1   
6      6  qp02zy   48  0.042247              3              1          1   

   Temperature  lat_scaled  lon_scaled  ...  hour  minute  \
1    31.104565   -1.985803   -0.752024  ...     0       0   
2    25.919267   -1.985803   -0.538873  ...     0       0   
4    10.803667   -1.890593   -0.858600  ...     0       0   
5     8.446025   -1.890593   -0.752024  ...     0       0   
6    15.772408   -1.890593   -0.645449  ...     0       0   

   minutes_since_midnight  time_sin  time_cos  RoadType_Residential  \
1                       0       0.0       1.0                     1   
2                     

In [7]:
df = df.drop(columns=['Weather_Sunny', 'geohash'])
df = df.drop(
    columns=[
        "hour",
        "minute",
        "minutes_since_midnight",
        "distance_from_city_center"
    ]
)
df = df.drop(columns=["Index"])

In [8]:
df.head()

,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,lat_scaled,lon_scaled,time_sin,time_cos,RoadType_Residential,RoadType_Street,Weather_Rainy,Weather_Snowy
1,48,0.118507,3,1,1,31.104565,-1.985803,-0.752024,0.0,1.0,1,0,0,0
2,48,0.027132,1,0,0,25.919267,-1.985803,-0.538873,0.0,1.0,1,0,0,0
4,48,0.010819,1,0,0,10.803667,-1.890593,-0.858600,0.0,1.0,1,0,1,0
5,48,0.016262,2,0,1,8.446025,-1.890593,-0.752024,0.0,1.0,1,0,1,0
6,48,0.042247,3,1,1,15.772408,-1.890593,-0.645449,0.0,1.0,1,0,0,0


In [9]:


# ----------------------------------
# Prepare data
# ----------------------------------

X = df.drop(columns=['demand'])
y = df['demand']
# Convert bool columns from get_dummies to int if needed
X = X.astype(float)

# Add intercept
X.insert(0, 'intercept', 1)

X = X.values
y = y.values

# ----------------------------------
# Train-test split
# ----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ----------------------------------
# Classical Gram-Schmidt
# ----------------------------------

def gram_schmidt(A):
    n, m = A.shape

    Q = np.zeros((n, m))
    R = np.zeros((m, m))

    for j in range(m):
        v = A[:, j].copy()

        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v = v - R[i, j] * Q[:, i]

        R[j, j] = np.linalg.norm(v)

        if R[j, j] > 1e-12:
            Q[:, j] = v / R[j, j]

    return Q, R

# ----------------------------------
# QR decomposition
# ----------------------------------

Q, R = gram_schmidt(X_train)

# ----------------------------------
# Solve Rβ = Qᵀy
# ----------------------------------

beta = np.linalg.solve(
    R,
    Q.T @ y_train
)

# ----------------------------------
# Predictions
# ----------------------------------

y_pred = X_test @ beta

# ----------------------------------
# Evaluation
# ----------------------------------

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R²:", r2)

# ----------------------------------
# Coefficients
# ----------------------------------

coef_df = pd.DataFrame({
    'Feature': ['Intercept'] + list(df.drop(columns=['demand']).columns),
    'Coefficient': beta
})

print(coef_df)

RMSE: 0.0712662619555829
R²: 0.7576580335591048
                 Feature  Coefficient
0              Intercept     0.233672
1                    day     0.007773
2          NumberofLanes     0.000677
3          LargeVehicles    -0.000494
4              Landmarks    -0.000973
5            Temperature    -0.000015
6             lat_scaled    -0.001773
7             lon_scaled    -0.000367
8               time_sin     0.008900
9               time_cos    -0.005888
10  RoadType_Residential    -0.551641
11       RoadType_Street    -0.336472
12         Weather_Rainy    -0.000776
13         Weather_Snowy    -0.000406


In [10]:
train_df.groupby('RoadType')['demand'].mean()

RoadType
Highway        0.611802
Residential    0.057195
Street         0.273146
Name: demand, dtype: float64

In [11]:
corr = df.corr(numeric_only=True)['demand'].sort_values()

print(corr)

RoadType_Residential   -0.785505
time_cos               -0.074971
lat_scaled             -0.039358
Landmarks              -0.011456
lon_scaled             -0.006777
Weather_Snowy          -0.003597
Weather_Rainy           0.002247
Temperature             0.003522
day                     0.025465
time_sin                0.075504
LargeVehicles           0.192352
NumberofLanes           0.211251
RoadType_Street         0.293400
demand                  1.000000
Name: demand, dtype: float64


In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_base = df.drop(columns=['demand'])
y = df['demand']

base_model = LinearRegression()
base_model.fit(X_base, y)

base_r2 = r2_score(
    y,
    base_model.predict(X_base)
)

print("Base R²:", base_r2)

Base R²: 0.7576167844413828


In [14]:
interaction_results = []

for col1, col2 in combinations(X_base.columns, 2):

    interaction = X_base[col1] * X_base[col2]

    X_temp = X_base.copy()

    X_temp["interaction"] = interaction

    model = LinearRegression()

    model.fit(X_temp, y)

    r2 = r2_score(
        y,
        model.predict(X_temp)
    )

    interaction_results.append({
        "feature": f"{col1}_x_{col2}",
        "r2_gain": r2 - base_r2
    })

interaction_results = pd.DataFrame(
    interaction_results
)

interaction_results = interaction_results.sort_values(
    "r2_gain",
    ascending=False
)

print(interaction_results.head(20))

                                 feature   r2_gain
59     lon_scaled_x_RoadType_Residential  0.002046
53     lat_scaled_x_RoadType_Residential  0.001500
68       time_cos_x_RoadType_Residential  0.000779
50               lat_scaled_x_lon_scaled  0.000372
64       time_sin_x_RoadType_Residential  0.000336
16            NumberofLanes_x_lon_scaled  0.000313
26            LargeVehicles_x_lon_scaled  0.000303
8             day_x_RoadType_Residential  0.000253
15            NumberofLanes_x_lat_scaled  0.000247
28              LargeVehicles_x_time_cos  0.000188
18              NumberofLanes_x_time_cos  0.000176
57                 lon_scaled_x_time_sin  0.000174
25            LargeVehicles_x_lat_scaled  0.000149
58                 lon_scaled_x_time_cos  0.000139
5                       day_x_lon_scaled  0.000105
1                    day_x_LargeVehicles  0.000077
65            time_sin_x_RoadType_Street  0.000064
13             NumberofLanes_x_Landmarks  0.000064
46    Temperature_x_RoadType_Re

In [15]:
geo_mean = train_df.groupby("geohash")["demand"].mean()

train_df["geo_mean_demand"] = train_df["geohash"].map(geo_mean)

In [16]:
geo_mean.head()

geohash
qp02yc    0.017716
qp02yf    0.029433
qp02yy    0.002902
qp02yz    0.036012
qp02z1    0.039774
Name: demand, dtype: float64

In [19]:


# Create directory
Path("../data/features").mkdir(parents=True, exist_ok=True)

# Save dataframe
df.to_csv(
    "../data/features/train_numeric_features.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!
